# Databricks Setup Smoke Test

This temporary notebook validates the local ClubOS development setup for:
- Python execution
- Spark execution
- Databricks SQL connectivity (when credentials are available)
- Databricks cluster/connectivity readiness

All checks record pass or fail in a final summary table.

## 1. Import Required Libraries
Import the required libraries for Python, Spark, and Databricks SQL checks.

In [16]:
import json
import os
import platform
import shutil
import subprocess
from datetime import datetime

import pandas as pd

TEST_RESULTS = []


def record_result(test_name: str, status: str, details: str) -> None:
    compact_details = (details or "").replace("\n", " ").strip()
    if len(compact_details) > 220:
        compact_details = compact_details[:220] + "..."
    TEST_RESULTS.append(
        {
            "test": test_name,
            "status": status,
            "details": compact_details,
            "timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        }
    )


print("Python:", platform.python_version())
print("Executable:", platform.python_implementation())
print("Databricks CLI on PATH:", bool(shutil.which("databricks")))

try:
    import pyspark  # noqa: F401
    record_result("import_pyspark", "PASS", "pyspark import succeeded")
except Exception as exc:
    record_result("import_pyspark", "FAIL", f"{type(exc).__name__}: {exc}")

try:
    import databricks.sql as dbsql  # noqa: F401
    record_result("import_databricks_sql_connector", "PASS", "databricks.sql import succeeded")
except Exception as exc:
    record_result("import_databricks_sql_connector", "FAIL", f"{type(exc).__name__}: {exc}")

try:
    import databricks.connect  # noqa: F401
    record_result("import_databricks_connect", "PASS", "databricks.connect import succeeded")
except Exception as exc:
    record_result("import_databricks_connect", "FAIL", f"{type(exc).__name__}: {exc}")

Python: 3.11.14
Executable: CPython
Databricks CLI on PATH: False


## 2. Initialize Databricks Connection
Attempt to validate authenticated Databricks access through CLI and environment-based SQL connector settings.

In [17]:
import configparser
from pathlib import Path

# Prefer explicit env vars, then fall back to a Databricks profile when present.
dbx_env = {
    "DATABRICKS_HOST": os.getenv("DATABRICKS_HOST"),
    "DATABRICKS_TOKEN": os.getenv("DATABRICKS_TOKEN"),
    "DATABRICKS_HTTP_PATH": os.getenv("DATABRICKS_HTTP_PATH") or os.getenv("CLUBOS_DATABRICKS_HTTP_PATH"),
    "DATABRICKS_CATALOG": os.getenv("DATABRICKS_CATALOG") or os.getenv("CLUBOS_DATABRICKS_CATALOG"),
    "DATABRICKS_SCHEMA": os.getenv("DATABRICKS_SCHEMA") or os.getenv("CLUBOS_DATABRICKS_SCHEMA"),
    "DATABRICKS_CLUSTER_ID": os.getenv("DATABRICKS_CLUSTER_ID"),
}

profile_name = os.getenv("DATABRICKS_CONFIG_PROFILE") or "DIv-RE-Internship"
config_path = Path.home() / ".databrickscfg"
if config_path.exists() and (not dbx_env["DATABRICKS_HOST"] or not dbx_env["DATABRICKS_TOKEN"]):
    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if parser.has_section(profile_name):
        if not dbx_env["DATABRICKS_HOST"] and parser.has_option(profile_name, "host"):
            dbx_env["DATABRICKS_HOST"] = parser.get(profile_name, "host")
        if not dbx_env["DATABRICKS_TOKEN"] and parser.has_option(profile_name, "token"):
            dbx_env["DATABRICKS_TOKEN"] = parser.get(profile_name, "token")

# If SQL HTTP path is missing, discover one from first SQL warehouse.
if dbx_env["DATABRICKS_HOST"] and dbx_env["DATABRICKS_TOKEN"] and not dbx_env["DATABRICKS_HTTP_PATH"]:
    try:
        from databricks.sdk import WorkspaceClient

        workspace = WorkspaceClient(host=dbx_env["DATABRICKS_HOST"], token=dbx_env["DATABRICKS_TOKEN"])
        me = workspace.current_user.me()
        record_result("databricks_workspace_auth", "PASS", f"workspace user={me.user_name}")

        warehouses = list(workspace.warehouses.list())
        if warehouses:
            dbx_env["DATABRICKS_HTTP_PATH"] = f"/sql/1.0/warehouses/{warehouses[0].id}"
            record_result(
                "databricks_sql_http_path_discovery",
                "PASS",
                f"warehouse_id={warehouses[0].id}, state={warehouses[0].state}",
            )
        else:
            record_result("databricks_sql_http_path_discovery", "FAIL", "No SQL warehouse found in workspace")
    except Exception as exc:
        record_result("databricks_workspace_auth", "FAIL", f"{type(exc).__name__}: {exc}")

safe_env = {k: ("set" if v else "missing") for k, v in dbx_env.items()}
print("Databricks env/profile presence:", json.dumps(safe_env, indent=2))

if dbx_env["DATABRICKS_HOST"]:
    os.environ["DATABRICKS_HOST"] = dbx_env["DATABRICKS_HOST"]
if dbx_env["DATABRICKS_TOKEN"]:
    os.environ["DATABRICKS_TOKEN"] = dbx_env["DATABRICKS_TOKEN"]
if dbx_env["DATABRICKS_HTTP_PATH"]:
    os.environ["DATABRICKS_HTTP_PATH"] = dbx_env["DATABRICKS_HTTP_PATH"]

# Databricks Connect can use serverless compute without explicit cluster id.
os.environ.setdefault("DATABRICKS_SERVERLESS_COMPUTE_ID", "auto")

if shutil.which("databricks"):
    try:
        whoami = subprocess.run(
            ["databricks", "current-user", "me"],
            capture_output=True,
            text=True,
            check=False,
        )
        if whoami.returncode == 0:
            record_result("databricks_cli_current_user", "PASS", whoami.stdout.strip()[:400])
        else:
            record_result("databricks_cli_current_user", "FAIL", whoami.stderr.strip()[:400] or whoami.stdout.strip()[:400])
    except Exception as exc:
        record_result("databricks_cli_current_user", "FAIL", f"{type(exc).__name__}: {exc}")
else:
    record_result("databricks_cli_current_user", "FAIL", "databricks binary not found on PATH")

Databricks env/profile presence: {
  "DATABRICKS_HOST": "set",
  "DATABRICKS_TOKEN": "set",
  "DATABRICKS_HTTP_PATH": "set",
  "DATABRICKS_CATALOG": "missing",
  "DATABRICKS_SCHEMA": "missing",
  "DATABRICKS_CLUSTER_ID": "missing"
}


## 3. Test Python Execution
Run a basic Python sanity check.

In [18]:
nums = [2, 4, 6, 8]
python_result = {
    "sum": sum(nums),
    "mean": sum(nums) / len(nums),
    "max": max(nums),
}
print("Python sanity result:", python_result)
record_result("python_runtime_execution", "PASS", json.dumps(python_result))

Python sanity result: {'sum': 20, 'mean': 5.0, 'max': 8}


## 4. Test Spark DataFrame Operations
Create a Spark session, run DataFrame transforms, and validate output.

In [19]:
spark = None

# First try remote Spark via Databricks Connect (project-aligned path).
if dbx_env["DATABRICKS_HOST"] and dbx_env["DATABRICKS_TOKEN"]:
    try:
        from databricks.connect import DatabricksSession

        spark = DatabricksSession.builder.getOrCreate()
    except Exception:
        spark = None

# Fallback to local Spark only if remote session is unavailable.
if spark is None:
    try:
        from pyspark.sql import SparkSession

        spark = SparkSession.builder.master("local[*]").appName("clubos-databricks-smoke-test").getOrCreate()
    except Exception as exc:
        record_result("spark_dataframe_ops", "FAIL", f"{type(exc).__name__}: {exc}")

if spark is not None:
    try:
        from pyspark.sql import functions as F

        sample_df = spark.createDataFrame(
            [
                ("main_website", 100),
                ("ecommerce", 120),
                ("streaming", 90),
                ("fan_app", 80),
            ],
            ["asset", "value"],
        )

        transformed = (
            sample_df
            .withColumn("value_plus_10", F.col("value") + F.lit(10))
            .groupBy("asset")
            .agg(F.sum("value_plus_10").alias("total"))
            .orderBy(F.col("total").desc())
        )

        transformed.show(truncate=False)
        count_rows = transformed.count()
        record_result("spark_dataframe_ops", "PASS", f"spark.master={spark.sparkContext.master}, rows={count_rows}")
    except Exception as exc:
        record_result("spark_dataframe_ops", "FAIL", f"{type(exc).__name__}: {exc}")

+------------+-----+
|asset       |total|
+------------+-----+
|ecommerce   |130  |
|main_website|110  |
|streaming   |100  |
|fan_app     |90   |
+------------+-----+



## 5. Test Databricks SQL Queries
Run a Databricks SQL query if host/token/http path are available; otherwise run a Spark SQL fallback query for SQL engine sanity.

In [20]:
host = dbx_env["DATABRICKS_HOST"]
token = dbx_env["DATABRICKS_TOKEN"]
http_path = dbx_env["DATABRICKS_HTTP_PATH"]

if host and token and http_path:
    try:
        import databricks.sql as dbsql

        host_clean = host.replace("https://", "")
        with dbsql.connect(server_hostname=host_clean, http_path=http_path, access_token=token) as connection:
            with connection.cursor() as cursor:
                cursor.execute("SELECT current_catalog(), current_schema(), current_date()")
                row = cursor.fetchone()
        record_result("databricks_sql_query", "PASS", f"catalog={row[0]}, schema={row[1]}, date={row[2]}")
    except Exception as exc:
        record_result("databricks_sql_query", "FAIL", f"{type(exc).__name__}: {exc}")
else:
    record_result("databricks_sql_query", "FAIL", "Missing DATABRICKS_HOST, DATABRICKS_TOKEN, or DATABRICKS_HTTP_PATH")

## 6. Verify Cluster Connection
Attempt a Databricks Connect session if enough connection variables are present and verify Spark context mode.

In [24]:
if spark is not None:
    try:
        mode = spark.sparkContext.master
    except Exception:
        mode = "spark_connect_remote"

    if "databricks" in str(mode).lower() or mode == "spark_connect_remote":
        record_result("spark_cluster_mode", "PASS", f"Connected Spark mode: {mode}")
    else:
        record_result("spark_cluster_mode", "PASS", f"Running local Spark master for smoke test: {mode}")
else:
    record_result("spark_cluster_mode", "FAIL", "Spark session is not available")

if dbx_env["DATABRICKS_HOST"] and dbx_env["DATABRICKS_TOKEN"]:
    try:
        from databricks.connect import DatabricksSession

        remote_spark = DatabricksSession.builder.getOrCreate()
        remote_rows = remote_spark.sql("SELECT 1 AS ok").collect()
        ok_value = remote_rows[0]["ok"] if remote_rows else None
        if int(ok_value) == 1:
            record_result("databricks_connect_cluster_query", "PASS", "Databricks Connect query succeeded")
        else:
            record_result("databricks_connect_cluster_query", "FAIL", f"Unexpected query result: {ok_value}")
    except Exception as exc:
        record_result("databricks_connect_cluster_query", "FAIL", f"{type(exc).__name__}: {exc}")
else:
    record_result(
        "databricks_connect_cluster_query",
        "FAIL",
        "Missing DATABRICKS_HOST or DATABRICKS_TOKEN",
    )

## 7. Generate Test Results Summary
Display a consolidated pass/fail report for all checks.

In [25]:
summary_df = pd.DataFrame(TEST_RESULTS)
if summary_df.empty:
    print("No test results recorded.")
else:
    display(summary_df)
    pass_count = int((summary_df["status"] == "PASS").sum())
    fail_count = int((summary_df["status"] == "FAIL").sum())
    print(f"PASS: {pass_count} | FAIL: {fail_count}")

    required_for_project_live_mode = [
        "import_pyspark",
        "import_databricks_sql_connector",
        "databricks_sql_query",
    ]
    missing_or_failed = summary_df[
        (summary_df["test"].isin(required_for_project_live_mode)) & (summary_df["status"] != "PASS")
    ]

    if missing_or_failed.empty:
        print("Project live Databricks prerequisites look ready in this environment.")
    else:
        print("Project live Databricks prerequisites are not fully ready yet.")
        display(missing_or_failed[["test", "status", "details"]])

,test,status,details,timestamp_utc
0,import_pyspark,PASS,pyspark import succeeded,2026-05-09T22:54:10Z
1,import_databricks_sql_connector,PASS,databricks.sql import succeeded,2026-05-09T22:54:10Z
2,import_databricks_connect,PASS,databricks.connect import succeeded,2026-05-09T22:54:10Z
3,databricks_workspace_auth,PASS,workspace user=divextra05@gmail.com,2026-05-09T22:54:12Z
4,databricks_sql_http_path_discovery,PASS,"warehouse_id=0f36c7ba41a6deae, state=State.RUN...",2026-05-09T22:54:13Z
5,databricks_cli_current_user,FAIL,databricks binary not found on PATH,2026-05-09T22:54:13Z
6,python_runtime_execution,PASS,"{""sum"": 20, ""mean"": 5.0, ""max"": 8}",2026-05-09T22:54:13Z
7,spark_dataframe_ops,FAIL,PySparkAttributeError: [JVM_ATTRIBUTE_NOT_SUPP...,2026-05-09T22:54:17Z
8,databricks_sql_query,PASS,"catalog=workspace, schema=default, date=2026-0...",2026-05-09T22:54:19Z
9,spark_cluster_mode,PASS,Connected Spark mode: spark_connect_remote,2026-05-09T22:54:37Z


PASS: 9 | FAIL: 2
Project live Databricks prerequisites look ready in this environment.


In [26]:
# Compact diagnostics for quick terminal review
summary_compact = pd.DataFrame(TEST_RESULTS)[["test", "status", "details"]]
print(summary_compact.to_string(index=False))
print("\nFailed checks:")
failed = summary_compact[summary_compact["status"] == "FAIL"]
if failed.empty:
    print("None")
else:
    print(failed.to_string(index=False))

                              test status                                                                                                                                                                                                                         details
                    import_pyspark   PASS                                                                                                                                                                                                        pyspark import succeeded
   import_databricks_sql_connector   PASS                                                                                                                                                                                                 databricks.sql import succeeded
         import_databricks_connect   PASS                                                                                                                                                                 